# Semantic Document Search
### *Finding meaning beyond keywords — powered by vector embeddings & FAISS*

---

> **Semantic search** understands the *intent* and *context* of a query, not just its words.  
> By encoding text as high-dimensional vectors, we can compare documents by **meaning** rather than exact wording.

---

## How it works

| Step | Stage | Description |
|------|-------|-------------|
| 1 | **Document Ingestion** | PDF, DOCX and TXT files are loaded and converted to plain text |
| 2 | **Chunking** | Each document is split into smaller, meaningful segments |
| 3 | **Embedding Generation** | Text chunks become dense numerical vectors via `SentenceTransformer` |
| 4 | **FAISS Indexing** | Vectors are stored in a FAISS index for blazing-fast retrieval |
| 5 | **Query Encoding** | The user's query is projected into the same vector space |
| 6 | **Similarity Search** | FAISS finds the top-k matches using **cosine similarity** |
| 7 | **Results Display** | The most relevant document snippets are returned |

---

**Tech stack:** `FAISS` · `SentenceTransformers` · `PyMuPDF` · `PyPDF2` · `python-docx`

## Step 1: Install Dependencies

In [1]:
!pip install faiss-cpu sentence-transformers python-docx PyMuPDF PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 68.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.1 MB/s eta 0:00:00


## Step 2 — Import Libraries

Load all necessary modules for file reading, vector operations, and semantic search.

In [2]:
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
from docx import Document

## Step 3 — Extract Text from Documents

A unified reader that handles **three file formats**:

| Format | Library used |
|--------|-------------|
| `.pdf` | `PyPDF2` — reads each page |
| `.docx` | `python-docx` — iterates over paragraphs |
| `.txt` | built-in `open()` |

In [3]:
def extract_text_from_file(file_path):

    text = ""
    if file_path.endswith(".pdf"):
        reader = PdfReader(file_path)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    elif file_path.endswith(".docx"):
        doc = Document(file_path)
        for para in doc.paragraphs:
            text += para.text + "\n"
    elif file_path.endswith(".txt"):
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
    return text.strip()

## Step 4 — Split Text into Chunks

Long documents are sliced into **overlapping word windows** (default: 300 words per chunk).  
Smaller chunks improve retrieval precision — the model focuses on a tight, relevant passage rather than an entire page.

In [4]:
def chunk_text(text, chunk_size=300):

    words = text.split()
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

## Step 5 — Load & Process Documents

Scans the target folder, extracts text from every supported file, and builds two parallel lists:
- `documents` — the actual text chunks
- `doc_sources` — the filename each chunk came from (for traceability in results)

In [5]:
folder_path = "/kaggle/input/datasets/nelsonvikou/toto-info"
documents = []
doc_sources = []

for file in os.listdir(folder_path):
    if file.endswith((".pdf", ".docx", ".txt")):
        path = os.path.join(folder_path, file)
        print(f"Reading file: {file}")
        content = extract_text_from_file(path)
        chunks = chunk_text(content)
        documents.extend(chunks)
        doc_sources.extend([file] * len(chunks))

print(
    f"\nLoaded {len(documents)} text chunks from {len(os.listdir(folder_path))} files.")

Reading file: SemanticDocumentSearch.txt
Reading file: SemanticDocumentSearch.docx
Reading file: SemanticDocumentSearch.pdf

Loaded 3 text chunks from 4 files.


## Step 6 — Generate Embeddings

Each text chunk is encoded into a **384-dimensional vector** using `all-MiniLM-L6-v2`, a fast and accurate sentence embedding model.  
Vectors are then **L2-normalized** so that dot product equals cosine similarity — required for FAISS `IndexFlatIP`.

In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print("\nGenerating embeddings... (this may take a minute)")

embeddings = model.encode(
    documents, convert_to_numpy=True, show_progress_bar=True)
embeddings = embeddings.astype('float32')
faiss.normalize_L2(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Generating embeddings... (this may take a minute)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (3, 384)


## Step 7 — Build the FAISS Index

`IndexFlatIP` performs an **exact inner product search** (= cosine similarity on normalized vectors).  
All chunk embeddings are added to the index — queries will be matched against this entire collection at millisecond speed.

In [7]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 3 vectors.


## Step 8 — Search Functions

Two helpers power the retrieval:

- **`clean_text()`** — strips Markdown symbols and normalizes whitespace for readable output
- **`semantic_search_best()`** — encodes the query, queries FAISS, filters by a `similarity_threshold`, and prints ranked results with source file, score, and a text preview

> **Tip:** increase `top_k` to see more candidates, or lower `similarity_threshold` to allow weaker matches.

In [8]:
import re
import textwrap


def clean_text(text):

    text = re.sub(r'[#=*`~_-]+', '', text)
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def semantic_search_best(query, top_k=1, wrap_width=100, similarity_threshold=0.35, snippet_length=300):

    query_embedding = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, top_k)

    print("\nTop Semantic Search Result(s):")
    print("=" * 120)

    results_shown = 0

    for rank, idx in enumerate(I[0]):
        score = D[0][rank]
        if score < similarity_threshold:
            continue

        snippet = clean_text(documents[idx])[:snippet_length]
        wrapped_snippet = textwrap.fill(snippet, width=wrap_width)

        print(f"\nRank {rank + 1}")
        print(f"Source File     : {doc_sources[idx]}")
        print(f"Similarity Score: {score:.4f}")
        print("-" * 120)
        print(f"Preview Snippet:\n{wrapped_snippet}")
        print("=" * 120)
        results_shown += 1

    if results_shown == 0:
        print("No strong semantic matches found for your query.")

## Step 9 — Run Semantic Search

Fire queries against the index. Each result shows:
- the **source document**, the **similarity score** (0 → 1, higher is better), and a **text preview**

The second call uses `top_k=3` to retrieve the **three most relevant passages**.

In [9]:
semantic_search_best("Semantic document search uses AI")


Top Semantic Search Result(s):

Rank 1
Source File     : SemanticDocumentSearch.pdf
Similarity Score: 0.7220
------------------------------------------------------------------------------------------------------------------------
Preview Snippet:
Semantic Document Search Semantic document search uses AI, natural language processing, and vector
embeddings to understand the meaning, context, and intent behind queries, rather than just matching
keywords. It converts text into numerical vectors to find semantically similar, relevant documents.
T


In [10]:
semantic_search_best("Semantic document search uses AI", top_k=3)


Top Semantic Search Result(s):

Rank 1
Source File     : SemanticDocumentSearch.pdf
Similarity Score: 0.7220
------------------------------------------------------------------------------------------------------------------------
Preview Snippet:
Semantic Document Search Semantic document search uses AI, natural language processing, and vector
embeddings to understand the meaning, context, and intent behind queries, rather than just matching
keywords. It converts text into numerical vectors to find semantically similar, relevant documents.
T

Rank 2
Source File     : SemanticDocumentSearch.docx
Similarity Score: 0.7046
------------------------------------------------------------------------------------------------------------------------
Preview Snippet:
Semantic Document Search Semantic document search uses AI, natural language processing, and vector
embeddings to understand the meaning, context, and intent behind queries, rather than just matching
keywords. It converts text into num